# Auto-MQT Colab

This notebook in Google Colab installs MQT-LLaVA, connects it to the Auto-MQT project code, and runs a tiny fixed-budget inference test.


In [1]:
# CUDA GPU
import torch, os, sys
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')
assert torch.cuda.is_available(), 'Switch Colab runtime to GPU before continuing.'

cuda: True
device: Tesla T4


## Install MQT-LLaVA

In [2]:
%cd /content
!test -d MQT-LLaVA || git clone https://github.com/gordonhu608/MQT-LLaVA.git
!pip install -q transformers==4.36.2 tokenizers==0.15.1 accelerate==0.21.0 sentencepiece==0.1.99 shortuuid einops==0.6.1 einops-exts==0.0.4 timm==0.6.13 protobuf peft
!pip install -q -e /content/MQT-LLaVA --no-deps

/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llava (pyproject.toml) ... done


In [3]:
# import MQT model
import transformers
print('transformers:', transformers.__version__)
sys.path.insert(0, '/content/MQT-LLaVA')
from llava.eval.run_llava import eval_model

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


transformers: 4.36.2


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [9]:
# upload project zip
from google.colab import files
uploaded = files.upload()

zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
assert zip_name is not None
!rm -rf /content/Project
!unzip -q -o "$zip_name" -d /content

# If the zip extracted as /content/Project, use it. If it extracted as /content/.../Project, adjust here.
PROJECT_DIR = '/content/Project'
assert os.path.isdir(PROJECT_DIR), 'Expected /content/Project after unzip. Check the zip structure.'
print('Project ready:', PROJECT_DIR)

Saving auto_mqt_project.zip to auto_mqt_project (1).zip
Project ready: /content/Project


## Configure Project Environment

In [10]:
os.environ['MQT_LLAVA_REPO'] = '/content/MQT-LLaVA'
os.environ['MQT_LLAVA_MODEL_PATH'] = 'gordonhu/MQT-LLaVA-7b'
os.environ['MQT_LLAVA_OFFLOAD_FOLDER'] = '/content/offload'
os.environ['MQT_LLAVA_BACKEND'] = 'persistent'
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
%cd $PROJECT_DIR
!python3 -m py_compile src/mqt_llava_adapter.py src/evaluate_token_policy.py src/prepare_datasets.py src/verify_manifest.py
print('Auto-MQT project import/compile ok')


/content/Project
Auto-MQT project import/compile ok


## Use Existing Manifest

If your zip included `data/manifests/eval.jsonl` and `data/images`, verify it here.

In [11]:
MANIFEST = 'data/manifests/eval.jsonl'
if os.path.exists(MANIFEST):
    !python3 src/verify_manifest.py --manifest data/manifests/eval.jsonl
else:
    print('No existing manifest found.')

manifest ok: data/manifests/eval.jsonl
rows: 2


## Hugging Face Subsets

Run this if you did not upload prepared manifests/images. Start with one dataset and 1-2 examples to confirm the model path.

In [7]:
# This may download dataset assets. If a configured HF mirror fails, edit configs/datasets.yaml or try another dataset name.
# !python3 src/prepare_datasets.py --datasets textvqa --train-limit 1 --eval-limit 1 --prompt-style none
# !python3 src/verify_manifest.py --manifest data/manifests/eval.jsonl
# MANIFEST = 'data/manifests/eval.jsonl'

## MQT-LLaVA Test


In [12]:
from pathlib import Path
from itertools import islice

tiny_manifest = Path('data/manifests/eval_tiny_1.jsonl')
source_manifest = Path(MANIFEST)
assert source_manifest.exists(), f'Manifest not found: {source_manifest}'
tiny_manifest.parent.mkdir(parents=True, exist_ok=True)
with source_manifest.open('r', encoding='utf-8') as src, tiny_manifest.open('w', encoding='utf-8') as dst:
    for line in islice(src, 1):
        dst.write(line)
print(tiny_manifest.read_text()[:500])

{"dataset": "textvqa", "split": "train", "example_id": "textvqa_train_142", "image": "data/images/textvqa/train/textvqa_train_142.jpg", "prompt": "what time is it currently?", "answer": "5:43 pm", "answers": ["5:43 pm", "5:35", "13", "5:43 p.m.", "5:35pm", "5:35 pm", "5:43"], "task": "ocr"}



In [13]:
# fixed low (36 tokens) budget
!mkdir -p results
!python3 src/evaluate_token_policy.py \
  --data data/manifests/eval_tiny_1.jsonl \
  --fixed-budget 36 \
  --prompt-style short \
  --out results/tiny_fixed_36.jsonl

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for

In [16]:
# fixed high (144 tokens) budget
!python3 src/evaluate_token_policy.py \
  --data data/manifests/eval_tiny_1.jsonl \
  --fixed-budget 144 \
  --prompt-style short \
  --out results/tiny_fixed_144.jsonl

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for

In [14]:
# results for 36 token budget
!cat results/tiny_fixed_36.jsonl

{"dataset": "textvqa", "split": "train", "example_id": "textvqa_train_142", "image": "data/images/textvqa/train/textvqa_train_142.jpg", "prompt": "what time is it currently?", "answer": "5:43 pm", "answers": ["5:43 pm", "5:35", "13", "5:43 p.m.", "5:35pm", "5:35 pm", "5:43"], "task": "ocr", "prediction": "The current time is 10:45.", "visual_tokens": 36, "latency_s": 68.21447317399998, "exact_match": 0.0}


In [17]:
# results for 144 token budget
!cat results/tiny_fixed_144.jsonl

{"dataset": "textvqa", "split": "train", "example_id": "textvqa_train_142", "image": "data/images/textvqa/train/textvqa_train_142.jpg", "prompt": "what time is it currently?", "answer": "5:43 pm", "answers": ["5:43 pm", "5:35", "13", "5:43 p.m.", "5:35pm", "5:35 pm", "5:43"], "task": "ocr", "prediction": "The current time is 5:32 PM according to the clock on the sign.", "visual_tokens": 144, "latency_s": 67.62954270800037, "exact_match": 0.0}


## Oracle Label A Small Training Subset

After fixed-budget inference works, run this to generate oracle labels. The script writes one row at a time and resumes automatically if Colab disconnects.


In [ ]:
!python3 src/oracle_labeling.py \
  --data data/manifests/train.jsonl \
  --out data/manifests/oracle_train_textvqa_small.jsonl \
  --budgets 36 64 144 256 \
  --score-key relaxed_match \
  --prompt-style short \
  --limit 10


In [ ]:
!tail -n 3 data/manifests/oracle_train_textvqa_small.jsonl


## Download Results

In [18]:
from google.colab import files
files.download('results/tiny_fixed_36.jsonl')
files.download('results/tiny_fixed_144.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>